%md
# 07 · Gold agent views
Label-free views over silver that form the agent's data contract. The MCP server reads only aml_gold — never aml_silver or aml_eval.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.aml_gold.transactions AS
SELECT
    t.txn_id,
    t.txn_timestamp,
    t.from_account_id,
    t.to_account_id,
    CAST(t.amount_paid AS DOUBLE) AS amount_paid,
    t.payment_currency,
    CAST(t.amount_paid AS DOUBLE) / fp.units_per_usd AS amount_paid_usd,
    CAST(t.amount_received AS DOUBLE) AS amount_received,
    t.receiving_currency,
    CAST(t.amount_received AS DOUBLE) / fr.units_per_usd AS amount_received_usd,
    t.payment_format,
    t.is_self_transfer
FROM workspace.aml_silver.transactions t
JOIN workspace.aml_gold.fx_rates fp ON t.payment_currency = fp.currency
JOIN workspace.aml_gold.fx_rates fr ON t.receiving_currency = fr.currency
""")

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.aml_gold.accounts AS
SELECT
    account_id,
    bank_id,
    bank_name,
    account_number,
    entity_id,
    entity_name
FROM workspace.aml_silver.accounts
""")

In [0]:
GOLD = "workspace.aml_gold"

txn_view = spark.table(f"{GOLD}.transactions")
acc_view = spark.table(f"{GOLD}.accounts")

txn_rows = txn_view.count()
acc_rows = acc_view.count()

print(f"Transactions view: {txn_rows:,}")
print(f"Accounts view:     {acc_rows:,}")

gold_tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {GOLD}").collect()]
print(f"Gold objects: {gold_tables}")

forbidden = []
for t in gold_tables:
    for c in spark.table(f"{GOLD}.{t}").columns:
        if "laundering" in c.lower() or "pattern" in c.lower():
            forbidden.append(f"{t}.{c}")

assert txn_rows == 5078345, "Transactions view lost rows (missing FX rate?)"
assert acc_rows == 518581, "Accounts view row count changed"
assert not forbidden, f"Label columns visible to the agent: {forbidden}"

print("Quality gate passed: all of aml_gold is label-free")